# ADNI1 → coronal slices for the five slice networks (GPU runtime)

Builds the ADNI1 counterpart of the OASIS-1 slice set, following `docs/prespecification_networks_adni.md`:
HD-BET skull stripping → FreeSurfer-style conform (1 mm, 256³, LIA) → white-matter peak at 110 → the same 24
coronal slices (indices 69–119 of the third axis) as 8-bit PNGs (`scripts/adni_slices.py`).

**Needs on Google Drive**
- `ADNI/ADNI1_Complete 1Yr 1.5T/ADNI/…` — the IDA download; only the 601 baseline images are read (`ADNI_ROOT` below).
- `ADNI/adni_network_cohort.csv` — built locally with `scripts/adni_network_cohort.py` (601 rows; contains ADNI IDs, keep it private).

**Writes to `ADNI/networks/`**: `adni_slices_fov_v1.tar` (the PNGs), `adni_hdbet_masks_v1.tar`, `qc_slices_fov.csv`,
`qc_montage_fov.png` and, if any, `failures_fov.csv`.

**Steps**: 1 setup → 2 config → 3 copy the T1s to local disk (reoriented to RAS) → 4 preview on six participants
(check the montage against the OASIS row before going on) → 5 HD-BET on all → 6 slices + QC → 7 archive to Drive.
Re-running skips finished steps. Expected time on an A100: 20–30 min to read 22 GB from Drive, 30–45 min for HD-BET
with test-time augmentation, about 10 min for slicing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, subprocess, sys
REPO = '/content/oasis1-cognitive-baseline'
REPO_REF = 'networks-v1'   # git tag pinned by docs/prespecification_networks_adni.md
if not os.path.exists(REPO):
    subprocess.check_call(['git', 'clone', '--quiet', 'https://github.com/Yuliawang520/oasis1-cognitive-baseline', REPO])
subprocess.check_call(['git', '-C', REPO, 'fetch', '--quiet', '--tags'])
subprocess.check_call(['git', '-C', REPO, 'checkout', '--quiet', REPO_REF])
print('repo at', subprocess.run(['git', '-C', REPO, 'rev-parse', '--short', 'HEAD'], capture_output=True, text=True).stdout.strip())
sys.path.insert(0, f'{REPO}/scripts')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True).stdout)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'hd-bet', 'nibabel'])

In [ ]:
DRIVE = '/content/drive/MyDrive'
ADNI_ROOT = f'{DRIVE}/ADNI/ADNI1_Complete 1Yr 1.5T/ADNI'   # the ADNI/ folder of the IDA download; adjust if your layout differs
COHORT_CSV = f'{DRIVE}/ADNI/adni_network_cohort.csv'
OUT_DRIVE = f'{DRIVE}/ADNI/networks'
OASIS_SLICES = f'{DRIVE}/alzheimer_anfis_outputs/baseline_densenet121_slices'   # for the visual comparison only
CENTER = 'fov'           # pre-specified primary; 'brain' only if the QC rule of the plan fires
PREVIEW_N = 6

T1_LOCAL, MASK_LOCAL = '/content/adni_t1', '/content/adni_masks'
SLICES_LOCAL = f'/content/adni_slices_{CENTER}'
for d in (OUT_DRIVE, T1_LOCAL, MASK_LOCAL):
    os.makedirs(d, exist_ok=True)

import pandas as pd
cohort = pd.read_csv(COHORT_CSV)
cohort['subject_id'] = cohort['subject_id'].astype(str)
print(len(cohort), 'participants;', cohort['label'].value_counts().to_dict(), '; folds', sorted(cohort['fold'].unique()))

In [ ]:
# 3. Locate the baseline T1 of every participant and copy it to local disk, reoriented to RAS (HD-BET expects
#    MNI-like axis order). The copy is the slow step (about 22 GB from Drive).
import nibabel as nib, numpy as np
from tqdm.notebook import tqdm

paths = {}
for _, r in cohort.iterrows():
    p = os.path.join(ADNI_ROOT, r['image_relpath'])
    paths[r['subject_id']] = p if os.path.exists(p) else None
missing = [s for s, p in paths.items() if p is None]
if missing:
    print(f'{len(missing)} images are not at their manifest path; scanning {ADNI_ROOT} by image id (slow on Drive) ...')
    wanted = dict(zip(cohort['image_id'].astype(str), cohort['subject_id']))
    for root, dirs, files in os.walk(ADNI_ROOT):
        iid = os.path.basename(root)
        if iid in wanted and paths[wanted[iid]] is None:
            nii = [f for f in files if f.endswith(('.nii', '.nii.gz'))]
            if nii:
                paths[wanted[iid]] = os.path.join(root, nii[0])
    missing = [s for s, p in paths.items() if p is None]
assert not missing, f'{len(missing)} images still missing, e.g. {missing[:5]}; check ADNI_ROOT'

for sid, p in tqdm(paths.items(), desc='T1 -> local RAS .nii.gz'):
    out = f'{T1_LOCAL}/{sid}.nii.gz'
    if os.path.exists(out):
        continue
    img = nib.as_closest_canonical(nib.load(p))
    nib.save(nib.Nifti1Image(np.asarray(img.dataobj, dtype=np.float32), img.affine), out)
print(len([f for f in os.listdir(T1_LOCAL) if f.endswith('.nii.gz')]), 'local T1 files')

In [ ]:
# 4. Preview: HD-BET and slices for the first PREVIEW_N participants, shown under one OASIS-1 subject.
import shutil
from pathlib import Path
from IPython.display import Image as IPImage, display
import adni_slices as S

preview = cohort['subject_id'].head(PREVIEW_N).tolist()
PREV_T1, PREV_MASK, PREV_SL = '/content/preview_t1', '/content/preview_masks', f'/content/preview_slices_{CENTER}'
os.makedirs(PREV_T1, exist_ok=True)
for sid in preview:
    if not os.path.exists(f'{PREV_T1}/{sid}.nii.gz'):
        shutil.copy(f'{T1_LOCAL}/{sid}.nii.gz', PREV_T1)
subprocess.run(['hd-bet', '-i', PREV_T1, '-o', PREV_MASK, '-device', 'cuda', '--save_bet_mask', '--no_bet_image'], check=True)
print('masks:', sorted(os.listdir(PREV_MASK)))
pd.DataFrame({'subject_id': preview}).to_csv('/content/preview_cohort.csv', index=False)
subprocess.run([sys.executable, f'{REPO}/scripts/adni_slices.py', '--cohort', '/content/preview_cohort.csv',
                '--t1-dir', PREV_T1, '--mask-dir', PREV_MASK, '--out-dir', PREV_SL, '--center', CENTER], check=True)
print(pd.read_csv(f'{PREV_SL}/qc_slices_{CENTER}.csv').to_string())

rows = []
if os.path.isdir(OASIS_SLICES):
    oasis_subject = sorted(d for d in os.listdir(OASIS_SLICES) if d.startswith('OAS1'))[:1]
    S.qc_montage(Path(OASIS_SLICES), oasis_subject, Path('/content/preview_oasis_row.png'))
    rows.append(np.asarray(__import__('PIL.Image', fromlist=['Image']).open('/content/preview_oasis_row.png')))
    print('top row: OASIS-1 subject', oasis_subject[0], '(reference); other rows: ADNI1')
S.qc_montage(Path(PREV_SL), preview, Path('/content/preview_adni_rows.png'))
rows.append(np.asarray(__import__('PIL.Image', fromlist=['Image']).open('/content/preview_adni_rows.png')))
__import__('PIL.Image', fromlist=['Image']).fromarray(np.concatenate(rows, axis=0)).save('/content/preview_compare.png')
display(IPImage('/content/preview_compare.png', width=1400))

**Check before continuing:** the ADNI rows must show the same plane and coverage as the OASIS row (coronal slices from
the cerebellum/occipital lobes forward to the frontal horns), skull-stripped, with similar grey/white contrast.
If a row is cut off, dark or still has scalp, stop and report it.

In [ ]:
# 5. HD-BET on all participants (masks only). A finished mask archive on Drive is reused instead of re-running.
MASK_TAR = f'{OUT_DRIVE}/adni_hdbet_masks_v1.tar'
n_masks = lambda: len([f for f in os.listdir(MASK_LOCAL) if f.endswith('_bet.nii.gz')])
if n_masks() < len(cohort) and os.path.exists(MASK_TAR):
    shutil.unpack_archive(MASK_TAR, MASK_LOCAL)
if n_masks() < len(cohort):
    subprocess.run(['hd-bet', '-i', T1_LOCAL, '-o', MASK_LOCAL, '-device', 'cuda', '--save_bet_mask', '--no_bet_image'], check=True)
    shutil.make_archive(MASK_TAR[:-4], 'tar', MASK_LOCAL)
print(n_masks(), 'masks')

In [ ]:
# 6. Slices + QC for all participants.
subprocess.run([sys.executable, f'{REPO}/scripts/adni_slices.py', '--cohort', COHORT_CSV, '--t1-dir', T1_LOCAL,
                '--mask-dir', MASK_LOCAL, '--out-dir', SLICES_LOCAL, '--center', CENTER], check=True)
qc = pd.read_csv(f'{SLICES_LOCAL}/qc_slices_{CENTER}.csv')
display(IPImage(f'{SLICES_LOCAL}/qc_montage_{CENTER}.png', width=1400))
print(qc[['n_png', 'wm_peak_raw', 'mask_volume_ml', 'brain_k_first', 'brain_k_last', 'window_inside_brain']].describe().round(2).to_string())
outside = (qc['window_inside_brain'] < 1).mean()
print(f'\nwindow leaves the brain in {outside:.1%} of participants; fallback rule fires above 10%: {outside > 0.10}')
print('participants with fewer than 24 slices:', int((qc['n_png'] < 24).sum()))

In [ ]:
# 7. Archive the slices and the QC files to Drive.
tar = shutil.make_archive(f'{OUT_DRIVE}/adni_slices_{CENTER}_v1', 'tar', SLICES_LOCAL)
for f in (f'qc_slices_{CENTER}.csv', f'qc_montage_{CENTER}.png', f'failures_{CENTER}.csv'):
    if os.path.exists(f'{SLICES_LOCAL}/{f}'):
        shutil.copy(f'{SLICES_LOCAL}/{f}', OUT_DRIVE)
print(tar, round(os.path.getsize(tar) / 1e6), 'MB')
print('send back:', f'{OUT_DRIVE}/qc_slices_{CENTER}.csv', 'and', f'{OUT_DRIVE}/qc_montage_{CENTER}.png')